# Lost in Handoff: clean baselines on a Colab A100

Reruns the clean baselines (no faults, no defenses) on Qwen3-8B through Ollama, after the researcher
fixes (read-only note, intake note, wrap-up call for empty or turn-limited handoffs, no-op tasks dropped):
- `single-clean`: the **single agent**
- `clean-D0`: the **3-agent pipeline with free-text handoffs**

Section 5 is an optional smoke test on 4 pilot-failure tasks; section 6 is the full baseline.
Runs go through `run_eval.py`, 4 at a time on the GPU. Results go to Google Drive. Sweeps resume, so
rerunning a cell skips trials that already have a record.

**Before you start:** Runtime → Change runtime type → **A100 GPU** → Save. Then Runtime → Run all.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Google Drive
Colab asks you to approve Drive access in a pop-up. Approve it yourself. Drive only stores the results
(and, optionally, a code zip), so a Colab disconnect never loses finished runs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/lost-in-handoff'
# v2 = after the researcher fixes; the pre-fix pilot stays in runs/colab_a100/baseline
OUT = f'{DRIVE_DIR}/runs/colab_a100/baseline_v2'
SMOKE_OUT = f'{DRIVE_DIR}/runs/colab_a100/smoke_v2'
!mkdir -p {OUT} {SMOKE_OUT} && ls -la {DRIVE_DIR}

## 2. Code: clone the branch from GitHub, fetch τ²-bench (same commit as local)
Set `BRANCH` to the branch to test. If `USE_ZIP` is True, the code comes from
`My Drive/lost-in-handoff/dist/lost-in-handoff-code.zip` instead (for code not pushed yet).

In [ ]:
import os, shutil, subprocess
BRANCH = 'main'  # or a teammate's branch, to test it before merging
USE_ZIP = False
shutil.rmtree('/content/lost-in-handoff', ignore_errors=True)
if USE_ZIP:
    ZIP = f'{DRIVE_DIR}/dist/lost-in-handoff-code.zip'
    print('using', ZIP)
    subprocess.run(['unzip', '-q', ZIP, '-d', '/content/lost-in-handoff'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '--branch', BRANCH,
                    'https://github.com/Shreepoorna1903/lost-in-handoff', '/content/lost-in-handoff'], check=True)
assert os.path.exists('/content/lost-in-handoff/pyproject.toml'), 'no pyproject.toml: wrong branch or zip'
%cd /content/lost-in-handoff
if not USE_ZIP:
    print('testing', subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
!git clone -q https://github.com/sierra-research/tau2-bench vendor/tau2-bench
!git -C vendor/tau2-bench checkout -q 57e54143025bec219b5726884651c98eede202e2
!git -C vendor/tau2-bench rev-parse HEAD
!ls

## 3. Python dependencies (uv honors the local tau2 source in pyproject.toml; pip does not)

In [ ]:
!pip install -q uv
!uv pip install --system -q -e ".[dev]"
!python -m pytest -q 2>&1 | tail -1

## 4. Ollama server (4 runs at a time) and Qwen3-8B
`OLLAMA_NUM_PARALLEL=4` lets 4 requests share the loaded model. `OLLAMA_KEEP_ALIVE=-1` keeps it in GPU
memory between calls.

In [ ]:
!apt-get -qq install -y zstd > /dev/null   # the Ollama installer needs zstd; Colab lacks it
!curl -fsSL https://ollama.com/install.sh | sh
import os, subprocess, time, urllib.request
PARALLEL = 4  # 2 on a T4; must match section 4b
env = dict(os.environ, OLLAMA_NUM_PARALLEL=str(PARALLEL), OLLAMA_KEEP_ALIVE='-1')
subprocess.Popen('ollama serve > /content/ollama.log 2>&1', shell=True, env=env)
for _ in range(30):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434'); print('Ollama is up'); break
    except Exception:
        time.sleep(2)
else:
    !tail -20 /content/ollama.log
    raise RuntimeError('Ollama did not start; see the log above')
!ollama pull qwen3:8b
!ollama list

## 4b. Run helpers (always run this cell)
`PARALLEL` runs share the one Ollama server: 4 on an A100 or L4, 2 on a T4 (16 GB).

In [ ]:
import glob, json, re, subprocess, time
MODEL, PARALLEL = 'qwen3:8b', 4  # 2 on a T4

def run_condition(cond, out, extra='', log=None):
    """Run one condition with run_eval.py in PARALLEL shards on the one Ollama server."""
    t0 = time.time()
    sink = f'>> {log}' if log else ''
    procs = [subprocess.Popen(
        f'python run_eval.py --condition {cond} --model {MODEL} --out {out} {extra} '
        f'--shard {k}/{PARALLEL} 2>&1 | grep -E --line-buffered "^task|Error|Traceback" {sink}', shell=True)
        for k in range(1, PARALLEL + 1)]
    for p in procs:
        p.wait()
    print(f'{cond}: done in {(time.time() - t0) / 60:.1f} min')

CLAIMED_DONE = re.compile(r'\b(has|have|was|were) been (processed|completed|initiated|made|done|'
                          r'exchanged|returned|cancelled|canceled|modified|refunded)|successfully', re.I)

## 5. Optional smoke test: the pipeline on 4 tasks that failed in the pilot
Skip this section to go straight to the full baseline. Tasks 0 and 2: the researcher claimed the changes
were already made. Tasks 3 and 13: it hit the turn limit. One trial each (about 5 minutes on an A100).
For each task the cell prints success, whether a wrap-up call fired, any "already done" wording in the
researcher's handoff, and the handoffs. **Check:** no handoff says an action was done, and the executor
makes write calls.

In [ ]:
run_condition('clean-D0', SMOKE_OUT, '--tasks 0,2,3,13')
for f in sorted(glob.glob(f'{SMOKE_OUT}/qwen3-8b/clean-D0/*.json')):
    r = json.load(open(f))
    if r.get('status') != 'ok':
        print(f, 'ERROR', r.get('error')); continue
    hs = r['state']['handoffs']
    res = next((h['content'] for h in hs if h['from'] == 'researcher'), '')
    writes = [t['tool'] for t in r['state']['tool_log'] if t.get('agent') == 'executor']
    print('=' * 100)
    print(f"task {r['task_id']}  success={r['success']}  wrap_ups={r['usage'].get('wrap_ups')}  "
          f"executor calls={writes}")
    print('claims done:', CLAIMED_DONE.findall(res) or 'none')
    print('--- intake handoff:\n', hs[0]['content'][:800])
    print('--- researcher handoff:\n', res[:1500])

## 6. Baseline sweep: single-clean, then clean-D0, 4 shards at a time, resumable
Runs the first `LIMIT` tasks (no-op tasks 10, 12, 50, 105 are already filtered out). Progress goes to
`OUT/sweep.log`. If the session dies, reconnect, rerun sections 1–4b, then rerun this cell.

In [ ]:
LIMIT = 10
LOG = f'{OUT}/sweep.log'
for cond in ['single-clean', 'clean-D0']:
    with open(LOG, 'a') as log:
        log.write(f'=== {cond} {time.strftime("%F %T")}\n')
    run_condition(cond, OUT, f'--limit {LIMIT}', log=LOG)
!tail -25 {LOG}

## 7. Summary

In [ ]:
!python score_eval.py {OUT}/qwen3-8b 2>&1 | grep -v DEBUG
for cond in ['single-clean', 'clean-D0']:
    res = {}
    for f in glob.glob(f'{OUT}/qwen3-8b/{cond}/*.json'):
        r = json.load(open(f))
        res[r['task_id']] = r.get('success')
    print(f'{cond:<14} solved', sorted((t for t, ok in res.items() if ok), key=int),
          ' failed', sorted((t for t, ok in res.items() if not ok), key=int))

## 8. Bring the results home
Download `My Drive/lost-in-handoff/runs/colab_a100/baseline_v2` (and `smoke_v2`) and put them in the
project's `runs/colab_a100/` folder, so they can be scored and committed next to the pre-fix pilot.